In [1]:
import numpy as np
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
books = pd.read_csv('/content/Books.csv')
ratings = pd.read_csv('/content/Ratings.csv')
users = pd.read_csv('/content/Users.csv')

/tmp/ipykernel_9489/3856252078.py:1: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  books = pd.read_csv('/content/Books.csv')


In [4]:
books.head()
ratings.head()
users.head()


,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN
3,4,"porto, v.n.gaia, portugal",17.0
4,5,"farnborough, hants, united kingdom",NaN


In [5]:
print(books.shape)
print(ratings.shape)
print(users.shape)


(271360, 8)
(1149780, 3)
(278858, 3)


In [6]:
books.isnull().sum()

,0
ISBN,0
Book-Title,0
Book-Author,2
Year-Of-Publication,0
Publisher,2
Image-URL-S,0
Image-URL-M,0
Image-URL-L,3


In [7]:
users.isnull().sum()

,0
User-ID,0
Location,0
Age,110762


In [8]:
ratings.isnull().sum()

,0
User-ID,0
ISBN,0
Book-Rating,0


In [9]:
books.duplicated().sum()

np.int64(0)

In [10]:
ratings.duplicated().sum()

np.int64(0)

In [11]:
users.duplicated().sum()

np.int64(0)

In [12]:
##popularity based recommendation system

In [13]:

ratings_with_name = ratings.merge(books, on='ISBN')

In [14]:
num_rating_df = ratings_with_name.groupby('Book-Title').count()['Book-Rating'].reset_index()
num_rating_df.rename(columns={'Book-Rating': 'num_ratings'}, inplace=True)
num_rating_df

,Book-Title,num_ratings
0,A Light in the Storm: The Civil War Diary of ...,4
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1
4,Beyond IBM: Leadership Marketing and Finance ...,1
...,...,...
241066,Ã?Â?lpiraten.,2
241067,Ã?Â?rger mit Produkt X. Roman.,4
241068,Ã?Â?sterlich leben.,1
241069,Ã?Â?stlich der Berge.,3


In [15]:
ratings_with_name['Book-Rating'] = pd.to_numeric(ratings_with_name['Book-Rating'], errors='coerce')
avg_rating_df = ratings_with_name.groupby('Book-Title').mean(numeric_only=True)['Book-Rating'].reset_index()
avg_rating_df.rename(columns={'Book-Rating': 'avg_rating'}, inplace=True)
avg_rating_df

,Book-Title,avg_rating
0,A Light in the Storm: The Civil War Diary of ...,2.250000
1,Always Have Popsicles,0.000000
2,Apple Magic (The Collector's series),0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,0.000000
...,...,...
241066,Ã?Â?lpiraten.,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,5.250000
241068,Ã?Â?sterlich leben.,7.000000
241069,Ã?Â?stlich der Berge.,2.666667


In [16]:
popular_df = num_rating_df.merge(avg_rating_df, on='Book-Title')
popular_df

,Book-Title,num_ratings,avg_rating
0,A Light in the Storm: The Civil War Diary of ...,4,2.250000
1,Always Have Popsicles,1,0.000000
2,Apple Magic (The Collector's series),1,0.000000
3,"Ask Lily (Young Women of Faith: Lily Series, ...",1,8.000000
4,Beyond IBM: Leadership Marketing and Finance ...,1,0.000000
...,...,...,...
241066,Ã?Â?lpiraten.,2,0.000000
241067,Ã?Â?rger mit Produkt X. Roman.,4,5.250000
241068,Ã?Â?sterlich leben.,1,7.000000
241069,Ã?Â?stlich der Berge.,3,2.666667


In [17]:
popular_df = popular_df[popular_df['num_ratings']>=250].sort_values('avg_rating', ascending=False).head(100)

In [18]:
popular_df = popular_df.merge(books, on='Book-Title').drop_duplicates('Book-Title')[['Book-Title','Book-Author','Image-URL-M','num_ratings','avg_rating']]

In [19]:
popular_df['Image-URL-M'][0]

'http://images.amazon.com/images/P/0439136350.01.MZZZZZZZ.jpg'

In [20]:
##collaborative Filtering Based Recommender System

In [21]:
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 400
padhe_likha_users = x[x].index
ratings_with_name['User-ID'].isin(padhe_likha_users)

,User-ID
0,False
1,False
2,False
3,False
4,False
...,...
1031131,False
1031132,False
1031133,False
1031134,False


In [22]:
x = ratings_with_name.groupby('User-ID').count()['Book-Rating'] > 400
padhe_likha_users = x[x].index

In [23]:
filtered_rating = ratings_with_name[ratings_with_name['User-ID'].isin(padhe_likha_users)]

In [24]:
y = filtered_rating.groupby('Book-Title').count()['Book-Rating'] >= 100
famous_books = y[y].index

In [25]:
final_ratings = filtered_rating = filtered_rating[filtered_rating['Book-Title'].isin(famous_books)]

In [26]:
pt = final_ratings.pivot_table(index='Book-Title', columns='User-ID', values='Book-Rating')

In [27]:
pt.fillna(0, inplace=True)

In [28]:
pt

User-ID,2276,3363,6251,6543,6575,7346,11601,11676,11993,12538,...,268932,269566,269719,270713,271284,274061,274308,275970,277427,278418
Book-Title,,,,,,,,,,,,,,,,,,,,,
1st to Die: A Novel,0.0,0.0,0.0,9.0,0.000000,0.0,0.0,9.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Painted House,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,9.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A Time to Kill,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Bridget Jones's Diary,0.0,0.0,0.0,0.0,2.666667,0.0,0.0,3.800000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Divine Secrets of the Ya-Ya Sisterhood: A Novel,0.0,0.0,0.0,0.0,8.000000,0.0,0.0,0.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Dreamcatcher,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,8.666667,0.0,0.0,...,8.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
Good in Bed,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,8.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Hannibal,0.0,0.0,6.0,0.0,0.000000,0.0,0.0,4.000000,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Harry Potter and the Chamber of Secrets (Book 2),0.0,0.0,0.0,0.0,0.000000,0.0,0.0,9.333333,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,8.0,9.0,0.0,0.0


In [29]:
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
similarity_scores = cosine_similarity(pt)

In [31]:
similarity_scores


array([[1.        , 0.11116443, 0.14375059, ..., 0.11523967, 0.18129851,
        0.066103  ],
       [0.11116443, 1.        , 0.22085899, ..., 0.07007731, 0.09789111,
        0.08441427],
       [0.14375059, 0.22085899, 1.        , ..., 0.0700147 , 0.05533445,
        0.        ],
       ...,
       [0.11523967, 0.07007731, 0.0700147 , ..., 1.        , 0.08772413,
        0.12694187],
       [0.18129851, 0.09789111, 0.05533445, ..., 0.08772413, 1.        ,
        0.07237767],
       [0.066103  , 0.08441427, 0.        , ..., 0.12694187, 0.07237767,
        1.        ]])

In [32]:
from numpy._core.defchararray import index
from scipy.spatial import distance
def recommend(book_name):
  #index fetch
  book_idx = np.where(pt.index==book_name)[0][0]
  similar_items = sorted(list(enumerate(similarity_scores[book_idx])),key=lambda x:x[1],reverse=True)[1:6]

  data = []
  for i in similar_items:
    item = []
    temp_df = books[books['Book-Title'] == pt.index[i[0]]]
    item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Title'].values))
    item.extend(list(temp_df.drop_duplicates('Book-Title')['Book-Author'].values))
    item.extend(list(temp_df.drop_duplicates('Book-Title')['Image-URL-M'].values))

    data.append(item)

  return data

In [33]:
recommend('The Firm')

[['The Rainmaker',
  'JOHN GRISHAM',
  'http://images.amazon.com/images/P/044022165X.01.MZZZZZZZ.jpg'],
 ['The Pelican Brief',
  'John Grisham',
  'http://images.amazon.com/images/P/0440214041.01.MZZZZZZZ.jpg'],
 ['The Client',
  'John Grisham',
  'http://images.amazon.com/images/P/038542471X.01.MZZZZZZZ.jpg'],
 ['The Chamber',
  'John Grisham',
  'http://images.amazon.com/images/P/0385424728.01.MZZZZZZZ.jpg'],
 ['A Time to Kill',
  'JOHN GRISHAM',
  'http://images.amazon.com/images/P/0440211727.01.MZZZZZZZ.jpg']]

In [34]:
pt.index[8]

'Harry Potter and the Chamber of Secrets (Book 2)'

In [35]:
import pickle
pickle.dump(popular_df,open('popular_df.pkl','wb'))

In [36]:
popular_df['Image-URL-M'][0]

'http://images.amazon.com/images/P/0439136350.01.MZZZZZZZ.jpg'

In [37]:
pickle.dump(pt,open('pt.pkl','wb'))
pickle.dump(books,open('books.pkl','wb'))
pickle.dump(similarity_scores,open('similarity_scores.pkl','wb'))
#